# 05 · Accuracy, evidence, and cost decision grid

A method that finds more supporting passages can still produce fewer correct
answers. Additional searches also consume calls and tokens, and their benefit
may depend on whether a question needs a single fact or a chain of facts.
Evaluating a method therefore requires more than an overall answer score.

This notebook brings answer quality, source coverage, failures, paired
uncertainty, and recorded cost into one comparison. Small invented cases make
the scoring rules explicit before the full MuSiQue/NQ results are examined by
corpus structure and query complexity. The grid keeps unavailable measurements
visible, while the experiments let you change the criterion used to judge a
method without changing the underlying results.

## 1. Evaluation across corpora and query complexity

The decision grid crosses **corpus/evidence structure** with **annotated query
complexity**. MuSiQue and NQ both contain Wikipedia passages; their multi-hop and
single-hop questions have different needs. MultiHop-RAG articles, with a rights record, would
add inference, comparison, and temporal query types. Rewrite wording is an
additional paired comparison, not a new corpus structure.

| Measure | Question it answers | Interpretation limit |
| --- | --- | --- |
| Answer EM / token F1, best alias | Does the prediction match an accepted answer? | Surface metrics can miss semantic equivalence |
| Recall@5/10/20 | What fraction of marked support appears in the ranking? | Alternative unmarked evidence may exist |
| All-supports@k | Does every marked support fit? | NQ can mark alternative sufficient contexts |
| Generation coverage | Which marked supports reached the final reader? | Coverage is not entailment |
| Valid citations | Do citations refer to exposed generation evidence? | Valid IDs can accompany a false claim |
| Abstentions / failures / stop reasons | Why was no useful answer produced? | Budget stops and provider failures are distinct |
| Calls, tokens, tools, service/wall time | What work was recorded? | Cache warmth and unfinished requests affect accounting |

For support set `G` and top-k IDs `R`, recall is `|G ∩ R| / |G|`;
all-supports is `1` only when `G` is a subset of `R`.
Answer metrics include every selected ID. Missing or failed cases must not
disappear from a denominator because a method found them difficult.

Both full evaluation splits contain answerable questions. NQ is a negative
control for the benefit of multi-hop reasoning, **not an unanswerable/null
control** like the contract examples in RI #1. An abstention receives zero
answer credit on these splits, even when refusing is appropriate because the
retrieved evidence is insufficient.


## 2. A worked example

### Separate answer correctness from evidence and operational success

Three invented cases below isolate scoring rules; they are not a benchmark.
One tests the missing bridge, one supplies an accepted answer alias, and one
contrasts a correct answer with incomplete evidence against a wrong answer with
complete evidence. Gold annotations enter only the scorer in this exercise.

**Predict first:** which arm has higher answer quality? Which has better
generation coverage? What happens when the alias case is a provider failure?

In [1]:
# ruff: noqa: E402 - resolve checkout imports before the first example
import os
import sys
from pathlib import Path


# True runs each model step on MODELS through the proxy and caches the result.
# False replays the reference results from RI #2's release with no model calls.
RUN_EXPENSIVE = False

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "aieng-advanced-rag/aieng").is_dir())
sys.path[:0] = [str(ROOT / "aieng-advanced-rag"), str(ROOT / "implementations/cross_document_synthesis")]
from synthesis_lab.tutorial import LAB, read_result, require_compute, show_table


require_compute()

from aieng.advanced_rag.models import choose_models


MODELS = choose_models(show=True)

from dataclasses import replace

from synthesis_lab.evaluation import paired_comparison, question_scores, summarize
from synthesis_lab.tutorial import synthetic_corpus


corpus, _, _ = synthetic_corpus()
template = corpus.questions[0]
questions = [
    replace(template, question_id="case::bridge"),
    replace(
        template,
        question_id="case::alias",
        question="Where was Mira born?",
        answers=("Oslo", "City of Oslo"),
        supporting_ids=("demo::10",),
        category="single-hop",
    ),
    replace(template, question_id="case::reader"),
]


def prediction(qid, answer, ids, *, calls=1, tokens=100):
    """Build an explicit synthetic prediction record, not a model execution."""
    return {
        "question_id": qid,
        "answer": answer,
        "abstained": not bool(answer),
        "retrieved_ids": ids,
        "generation_ids": ids,
        "citation_ids": ids if answer else [],
        "stop_reason": "answered" if answer else "insufficient",
        "usage": {"calls": calls, "total_tokens": tokens},
    }


control = [
    prediction("case::bridge", "", ["demo::00"]),
    prediction("case::alias", "City of Oslo", ["demo::10"]),
    prediction("case::reader", "Oslo", ["demo::00"]),
]
arm = [
    prediction("case::bridge", "Oslo", ["demo::00", "demo::10"], calls=4, tokens=800),
    {
        "question_id": "case::alias",
        "failure": {"type": "provider_outage"},
        "stop_reason": "execution_failure",
        "usage": {"calls": 2, "total_tokens": 200},
    },
    prediction("case::reader", "Bergen", ["demo::00", "demo::10"], calls=4, tokens=800),
]
show_table(
    ["Case", "Arm", "EM", "F1", "Generation all-supports", "Failure"],
    [
        [
            q.question_id,
            name,
            *[
                question_scores(q, row)[key]
                for key in ("exact_match", "token_f1", "generation_all_supports", "failure")
            ],
        ]
        for q, a, b in zip(questions, control, arm, strict=True)
        for name, row in (("control", a), ("variant", b))
    ],
)

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)


| Case | Arm | EM | F1 | Generation all-supports | Failure |
| --- | --- | --- | --- | --- | --- |
| case::bridge | control | 0.0000 | 0.0000 | 0.0000 | 0.0000 |
| case::bridge | variant | 1.0000 | 1.0000 | 1.0000 | 0.0000 |
| case::alias | control | 1.0000 | 1.0000 | 1.0000 | 0.0000 |
| case::alias | variant | 0.0000 | 0.0000 | 0.0000 | 1.0000 |
| case::reader | control | 1.0000 | 1.0000 | 0.0000 | 0.0000 |
| case::reader | variant | 0.0000 | 0.0000 | 1.0000 | 0.0000 |

The accepted alias receives full EM credit. A correct string with incomplete
evidence can score well on answers while failing generation coverage. Complete
evidence does not force the reader to answer correctly. The failed case receives
zero answer/retrieval credit, and the calls made before failure are retained.

In [2]:
show_table(
    ["Synthetic arm", "Denominator", "F1", "Generation all-supports", "Failure rate", "Calls"],
    [
        [
            name,
            scores["questions"],
            scores["metrics"]["token_f1"],
            scores["metrics"]["generation_all_supports"],
            scores["metrics"]["failure"],
            scores["usage"]["calls"],
        ]
        for name, scores in (
            ("control", summarize(questions, control)["overall"]),
            ("variant", summarize(questions, arm)["overall"]),
        )
    ],
)
try:
    summarize(questions, [r for r in arm if not r.get("failure")])
except ValueError as error:
    print("Dropping a failed case is rejected:", error)

| Synthetic arm | Denominator | F1 | Generation all-supports | Failure rate | Calls |
| --- | --- | --- | --- | --- | --- |
| control | 3 | 0.6667 | 0.3333 | 0.0000 | 3 |
| variant | 3 | 0.3333 | 0.6667 | 0.3333 | 10 |

Dropping a failed case is rejected: scores require exactly one record per selected question


## 3. Recorded results

### Check cohort and configuration before reading the ranking

The eight committed full reports each contain exactly 1,000 question outcomes.
They were replayed against pinned annotations before this summary was produced.
Loading the summary below displays those saved metrics; it is not a new replay.
The final answer template and five-passage budget match across core and graph methods.
The graph arms are atlas-rag HippoRAG 2 and LightRAG in `mix` mode, both over
RI #4's release `reference-ri4-bd05f7bb16c6`, which indexes all 11,656 passages.

### Evaluation data and fixed configuration

The table below identifies the saved full-corpus reference used in this section.
The MuSiQue reference searches all 11,656 passages for 1,000 original questions.
The NQ control searches 9,633 passages for its 1,000 questions. Model, encoder,
source revision, prompt identity, and evidence settings travel with the saved
reports. Loading these records makes no model calls and does not recompute the
measurements. New experiments must keep the controls fixed or identify the change.


In [3]:
saved = read_result("evaluation/musique_dense.json")
spec = saved["spec"]
assert saved["completed"] and saved["coverage_complete"] and len(saved["records"]) == 1000
show_table(
    ["Recorded setting", "Value"],
    [
        ["Answer model", spec["model"]],
        ["Generation endpoint", spec["base_url"]],
        ["Local encoder", spec["embedding"][0]],
        ["Encoder revision", spec["embedding"][1]],
        ["Corpus revision", spec["provenance"]["revision"]],
        ["Final passage budget", spec["bounds"]["generation_passages"]],
    ],
)
print("Answer-template SHA-256:", spec["answer_prompt_sha256"])
print("Saved configuration SHA-256:", saved["spec_sha256"])

| Recorded setting | Value |
| --- | --- |
| Answer model | Qwen3_6-35B-A3B |
| Generation endpoint | https://proxy.vectorinstitute.ai/v1 |
| Local encoder | BAAI/bge-small-en-v1.5 |
| Encoder revision | 5c38ec7c405ec4b44b94cc5a9bb96e735b38267a |
| Corpus revision | 5ec05b38deecc3318bb432c69865959c56058990 |
| Final passage budget | 5 |

Answer-template SHA-256: 517c503f1380aa40f6d20b1fc0b8f6538bbf367601e79d4bca0d316cea6eea0f
Saved configuration SHA-256: 40044ef66097a00af9d062567b74309de692c301c73ea0355ccdb6d3ff7ca620


In [4]:
summary = read_result("evaluation/graphrag/summary.json")
show_table(
    ["Corpus", "Method", "N", "EM", "F1", "Recall@5", "Generation all-supports", "Failures"],
    [
        [
            r["corpus"],
            r["method"],
            r["scores"]["overall"]["questions"],
            *[
                r["scores"]["overall"]["metrics"][k]
                for k in ("exact_match", "token_f1", "recall_at_5", "generation_all_supports", "failure")
            ],
        ]
        for r in summary["reports"]
    ],
)

| Corpus | Method | N | EM | F1 | Recall@5 | Generation all-supports | Failures |
| --- | --- | --- | --- | --- | --- | --- | --- |
| musique | dense | 1000 | 0.2010 | 0.2733 | 0.5060 | 0.1710 | 0.0000 |
| musique | decomposition | 1000 | 0.2060 | 0.2805 | 0.5540 | 0.1880 | 0.0000 |
| musique | ircot | 1000 | 0.4090 | 0.4854 | 0.6823 | 0.4300 | 0.0090 |
| nq | dense | 1000 | 0.5070 | 0.6302 | 0.6824 | 0.3510 | 0.0000 |
| nq | decomposition | 1000 | 0.5010 | 0.6249 | 0.6739 | 0.3440 | 0.0000 |
| nq | ircot | 1000 | 0.4840 | 0.6036 | 0.6812 | 0.3230 | 0.0060 |
| musique | hipporag2 | 1000 | 0.3140 | 0.4073 | 0.6254 | 0.3230 | 0.0000 |
| musique | lightrag | 1000 | 0.1950 | 0.2655 | 0.4663 | 0.1370 | 0.0000 |

### Inspect the two-axis grid

Rows combine evidence structure and annotated complexity. Measured cells show F1
and their denominator; unmeasured cells have no score. The correction cohort on the
shared 300 questions has its own grid and controls. Use its dense and graded
single-pass controls below for paired conclusions;
the grid juxtaposes workloads and is not a new pooled experiment.


In [5]:
METHODS = ("dense", "decomposition", "ircot", "hipporag2", "lightrag", "corrective")


def show_grid(metric="token_f1", include_pending=True):
    """Display actual grid cells; unavailable measurements never become zeroes."""
    by_row = {}
    for cell in summary["grid"]:
        key = (cell["corpus"], cell["corpus_structure"], cell["complexity"])
        by_row.setdefault(key, {})[cell["method"]] = cell
    rows = []
    for key, entries in by_row.items():
        if not include_pending and all(c["status"] == "pending" for c in entries.values()):
            continue
        values = []
        for method in METHODS:
            cell = entries.get(method)
            if cell is None:
                values.append("outside grid")
            elif cell["status"] == "pending":
                values.append("pending")
            else:
                flag = "; failures included" if cell["status"] == "measured_with_failures" else ""
                flag += "; separate cohort" if cell.get("comparison_set") else ""
                values.append(f"{cell['metrics'][metric]:.4f} (n={cell['questions']}{flag})")
        rows.append([*key, *values])
    show_table(["Corpus", "Evidence structure", "Query complexity", *METHODS], rows)


show_grid()

| Corpus | Evidence structure | Query complexity | dense | decomposition | ircot | hipporag2 | lightrag | corrective |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| musique | Wikipedia passages | 2-hop | 0.3223 (n=518) | 0.3120 (n=518) | 0.5738 (n=518) | 0.4745 (n=518) | 0.3048 (n=518) | pending |
| musique | Wikipedia passages | 3-hop | 0.2458 (n=316) | 0.2770 (n=316) | 0.4343 (n=316; failures included) | 0.3962 (n=316) | 0.2320 (n=316) | pending |
| musique | Wikipedia passages | 4-hop | 0.1727 (n=166) | 0.1887 (n=166) | 0.3071 (n=166; failures included) | 0.2188 (n=166) | 0.2069 (n=166) | pending |
| nq | Wikipedia passages | single-hop | 0.6302 (n=1000) | 0.6249 (n=1000) | 0.6036 (n=1000; failures included) | outside grid | outside grid | outside grid |
| multihop-rag | Full publisher articles | inference_query | pending | pending | pending | outside grid | outside grid | outside grid |
| multihop-rag | Full publisher articles | comparison_query | pending | pending | pending | outside grid | outside grid | outside grid |
| multihop-rag | Full publisher articles | temporal_query | pending | pending | pending | outside grid | outside grid | outside grid |

### Read paired differences, not two independent means

Each bootstrap draw resamples the same original IDs for both methods. The saved
95% percentile intervals use 5,000 resamples. Paired rewrite comparisons keep
plain and obfuscated versions together under the original ID. Each paired input
file records a version and digest so the wording comparison is reproducible.

In [6]:
show_table(
    ["Corpus", "Method vs dense", "Paired F1 difference", "95% interval", "Original IDs"],
    [
        [
            r["corpus"],
            r["method"],
            r["differences"]["token_f1"]["mean"],
            f"[{r['differences']['token_f1']['low']:.4f}, {r['differences']['token_f1']['high']:.4f}]",
            r["original_ids"],
        ]
        for r in summary["comparisons"]
    ],
)

| Corpus | Method vs dense | Paired F1 difference | 95% interval | Original IDs |
| --- | --- | --- | --- | --- |
| musique | decomposition | 0.0072 | [-0.0127, 0.0272] | 1000 |
| musique | ircot | 0.2122 | [0.1828, 0.2418] | 1000 |
| nq | decomposition | -0.0053 | [-0.0149, 0.0036] | 1000 |
| nq | ircot | -0.0266 | [-0.0415, -0.0115] | 1000 |
| musique | hipporag2 | 0.1340 | [0.1088, 0.1600] | 1000 |
| musique | lightrag | -0.0077 | [-0.0236, 0.0079] | 1000 |

IRCoT improves MuSiQue F1 by about 0.2122 and reduces NQ F1 by about 0.0266;
both intervals exclude zero. Decomposition's answer intervals cross zero.
These are recorded-response comparisons, not proof that the loop alone caused
every difference: provider aliases do not pin model weights, and cache replay
can span backend deployments.

### Account for the work represented by each result

Normalize recorded usage by the full denominator. Live calls and cache hits
remain separate. Summed per-question wall time is not elapsed batch duration
when workers overlap. Service time also reflects provider latency, and unknown
unfinished work means these totals are not complete billing estimates.

In [7]:
show_table(
    ["Corpus", "Method", "Live calls / q", "Cache hits / q", "Tokens / q", "Tool calls / q"],
    [
        [
            r["corpus"],
            r["method"],
            *[
                r["scores"]["overall"]["usage"][key] / r["scores"]["overall"]["questions"]
                for key in ("calls", "cache_hits", "total_tokens", "tool_calls")
            ],
        ]
        for r in summary["reports"]
    ],
)
reference = read_result("reference/execution_usage.json")
print(reference["note"])
show_table(
    ["Original MuSiQue execution", "Live calls", "Cache hits", "Recorded tokens"],
    [
        [r["method"], r["usage"]["calls"], r["usage"]["cache_hits"], r["usage"]["total_tokens"]]
        for r in reference["records"]
    ],
)

| Corpus | Method | Live calls / q | Cache hits / q | Tokens / q | Tool calls / q |
| --- | --- | --- | --- | --- | --- |
| musique | dense | 0.0000 | 1.0000 | 0.0000 | 0.0000 |
| musique | decomposition | 0.0000 | 2.0000 | 0.0000 | 0.0000 |
| musique | ircot | 6.2620 | 0.0730 | 15854.8190 | 4.6210 |
| nq | dense | 1.0000 | 0.0000 | 949.5850 | 0.0000 |
| nq | decomposition | 1.2610 | 0.7390 | 417.9370 | 0.0000 |
| nq | ircot | 4.4440 | 0.2200 | 9915.5430 | 2.7830 |
| musique | hipporag2 | 1.9980 | 0.0020 | 2523.8400 | 0.0000 |
| musique | lightrag | 1.9980 | 0.0010 | 1629.7600 | 0.0000 |

Recorded original execution usage, including its own cache hits. This is distinct from current warm replay and is not a cache-free cost estimate.


| Original MuSiQue execution | Live calls | Cache hits | Recorded tokens |
| --- | --- | --- | --- |
| dense | 999 | 1 | 892909 |
| decomposition | 1892 | 108 | 1003504 |

### Assess optional correction on its own matched cohort

The separate study on the shared 300 questions includes newly measured dense and graded
single-pass controls. Correction reuses RI #1's shared spine, grader, rewriter,
and policy with at most two attempts; thresholds were fixed before execution.
The graded control measures grading overhead without a retry. It shares exact
final-generation requests with dense, so its answers can be cached unchanged.

Compare correction with both controls using the same IDs. The cache namespace
starts separately from the historical core runs but is shared within this cohort;
report live calls and hits separately. The earlier 30-question pilot remains an
integration example in `outputs/examples/smoke/`. Do not mix the new controls
with the historical three-method cohort to calculate a paired gain.

The cohort is the bootcamp's shared 300 MuSiQue questions (150 two-hop, 75
three-hop, and 75 four-hop, drawn by `stratified_test_set`), the draw RI #4 and
RI #5 report on. It is not a full-split corrective result.


In [8]:
correction = summary["corrective_cohort"]
show_table(
    ["Cohort method", "N", "F1", "Recall@20", "Abstentions", "Failures", "Live calls", "Cache hits", "Tokens"],
    [
        [
            r["method"],
            r["scores"]["overall"]["questions"],
            *[r["scores"]["overall"]["metrics"][k] for k in ("token_f1", "recall_at_20", "abstained", "failure")],
            *[r["scores"]["overall"]["usage"][k] for k in ("calls", "cache_hits", "total_tokens")],
        ]
        for r in correction["reports"]
    ],
)
show_table(
    ["Method", "Control", "N", "Paired F1 difference", "95% interval", "Added live calls", "Added calls + hits"],
    [
        [
            r["method"],
            r["control"],
            r["questions"],
            r["differences"]["token_f1"]["mean"],
            f"[{r['differences']['token_f1']['low']:.4f}, {r['differences']['token_f1']['high']:.4f}]",
            r["added_calls"],
            r["added_calls_plus_cache_hits"],
        ]
        for r in correction["comparisons"]
        if r["method"] == "corrective"
    ],
)
paired_correction = [r for r in correction["comparisons"] if r["method"] == "corrective"]
print(
    "Positive paired F1 intervals against both controls:",
    all(r["quality_improvement_supported"] for r in paired_correction),
)
print("Assess added cost as well as the quality interval before recommending correction.")
show_table(
    ["Cohort method", "Workers", "Answering batch wall (s)", "Summed service (s)", "Summed question wall (s)"],
    [
        [
            r["method"],
            r.get("workers"),
            r.get("execution_wall_seconds"),
            r["scores"]["overall"]["usage"]["service_seconds"],
            r["scores"]["overall"]["summed_question_wall_seconds"],
        ]
        for r in correction["reports"]
    ],
)
print("Batch wall time covers answering; corpus/model/embedding setup is excluded.")

| Cohort method | N | F1 | Recall@20 | Abstentions | Failures | Live calls | Cache hits | Tokens |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| dense | 300 | 0.2742 | 0.6275 | 0.4267 | 0.0000 | 300 | 0 | 269117 |
| graded-single-pass | 300 | 0.2742 | 0.6275 | 0.4267 | 0.0000 | 1500 | 300 | 514287 |
| corrective | 300 | 0.2959 | 0.6250 | 0.3733 | 0.0033 | 1832 | 1544 | 715037 |

| Method | Control | N | Paired F1 difference | 95% interval | Added live calls | Added calls + hits |
| --- | --- | --- | --- | --- | --- | --- |
| corrective | dense | 300 | 0.0217 | [-0.0092, 0.0526] | 1532 | 3076 |
| corrective | graded-single-pass | 300 | 0.0217 | [-0.0092, 0.0526] | 332 | 1576 |

Positive paired F1 intervals against both controls: False
Assess added cost as well as the quality interval before recommending correction.


| Cohort method | Workers | Answering batch wall (s) | Summed service (s) | Summed question wall (s) |
| --- | --- | --- | --- | --- |
| dense | 6 | 215.4096 | 899.6220 | 1277.8231 |
| graded-single-pass | 6 | 1124.0008 | 3929.1250 | 6717.5880 |
| corrective | 6 | 1025.1983 | 2809.2020 | 6108.3042 |

Batch wall time covers answering; corpus/model/embedding setup is excluded.


The correction interval includes no improvement, so this study does not support
recommending its added work. “Not established” does not mean correction can never
help; a larger preregistered comparison could test a justified hypothesis.

## 4. Experiments

### Change the decision criterion

Edit `SELECTED_METRIC` to `token_f1`, `recall_at_5`, `generation_all_supports`,
`abstained`, or `failure`. For quality and coverage higher is better; for failure
lower is better. Abstention needs context, since over-answering can be harmful
in another corpus. Which method appears attractive under each objective?

`INCLUDE_PENDING=False` hides rows without any measurement, without converting
them into scores. Keep pending cells visible when discussing scope limitations.

In [9]:
SELECTED_METRIC = "generation_all_supports"
INCLUDE_PENDING = True
show_grid(SELECTED_METRIC, INCLUDE_PENDING)

| Corpus | Evidence structure | Query complexity | dense | decomposition | ircot | hipporag2 | lightrag | corrective |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| musique | Wikipedia passages | 2-hop | 0.2992 (n=518) | 0.3127 (n=518) | 0.6409 (n=518) | 0.5270 (n=518) | 0.2490 (n=518) | pending |
| musique | Wikipedia passages | 3-hop | 0.0506 (n=316) | 0.0823 (n=316) | 0.2753 (n=316; failures included) | 0.1487 (n=316) | 0.0253 (n=316) | pending |
| musique | Wikipedia passages | 4-hop | 0.0000 (n=166) | 0.0000 (n=166) | 0.0663 (n=166; failures included) | 0.0181 (n=166) | 0.0000 (n=166) | pending |
| nq | Wikipedia passages | single-hop | 0.3510 (n=1000) | 0.3440 (n=1000) | 0.3230 (n=1000; failures included) | outside grid | outside grid | outside grid |
| multihop-rag | Full publisher articles | inference_query | pending | pending | pending | outside grid | outside grid | outside grid |
| multihop-rag | Full publisher articles | comparison_query | pending | pending | pending | outside grid | outside grid | outside grid |
| multihop-rag | Full publisher articles | temporal_query | pending | pending | pending | outside grid | outside grid | outside grid |

### Inspect pairing on the synthetic cases

Edit the synthetic failure/correctness records above and rerun this cell.
The tiny cohort can teach paired arithmetic, not population inference. More
bootstrap samples reduce simulation noise; they do not create more questions or
fix a biased cohort. The production comparison retains 5,000 fixed resamples.

In [10]:
BOOTSTRAP_SAMPLES = 1_000
paired = paired_comparison(questions, arm, control, samples=BOOTSTRAP_SAMPLES)
print("Synthetic paired F1 difference and interval:", paired["differences"]["token_f1"])
print("Original paired IDs:", paired["original_ids"])
print("Added recorded calls:", paired["added_calls"])

Synthetic paired F1 difference and interval: {'mean': -0.3333333333333333, 'low': -1.0, 'high': 1.0}
Original paired IDs: 3
Added recorded calls: 7


## 5. Optional live execution

Set `RUN_EXPENSIVE = True` in the first code cell to enable this experiment.
The default `False` runs only the offline examples and recorded results.

This notebook's optional action is **score replay**, not a model run. It loads
pinned annotations, verifies question/digest/configuration/coverage contracts,
and recomputes metrics and paired statistics. Missing corpus files may need a
download; embeddings, hosted model calls, and graph construction are not used.

The inputs are the six core reports, the two full-split graph reports, and the
three matched corrective-cohort reports. Replay keeps the corrective controls separate. Replay output goes to
the ignored experiment directory. Use notebooks 01/03/04 for live method runs.

Live chat uses the shared model roles and Vector credential resolver. The
default family is Google; use `ADVANCED_RAG_MODEL_FAMILY=onprem` for Qwen
generation and extraction. Embeddings stay local. Set
`ADVANCED_RAG_SPEND_LIMIT_USD` before execution to limit spending. See the
[provider settings](GUIDE.md#models-credentials-and-spend). Recorded results
retain their original models; fresh comparisons require matched model roles.

In [11]:
RUN_REPLAY = RUN_EXPENSIVE
REPORT_PATHS = sorted(
    [
        *(LAB / "outputs/evaluation").glob("musique_*.json"),
        *(LAB / "outputs/evaluation").glob("nq_*.json"),
        LAB / "outputs/evaluation/graphrag/hipporag2.json",
        LAB / "outputs/evaluation/graphrag/lightrag.json",
    ]
)
if RUN_REPLAY:
    import subprocess

    subprocess.run(
        [
            sys.executable,
            str(LAB / "run_evaluation.py"),
            "--reports",
            *map(str, REPORT_PATHS),
            "--corrective-reports",
            *[
                str(LAB / "outputs/evaluation/corrective" / f"{method}.json")
                for method in ("dense", "graded-single-pass", "corrective")
            ],
            "--output",
            str(ROOT / "Arm Store/notebook-experiments/grid-replay.json"),
        ],
        cwd=ROOT,
        check=True,
        env=dict(os.environ, PYTHONPATH=str(ROOT / "aieng-advanced-rag")),
    )
else:
    print("Pinned-annotation replay skipped. Set RUN_EXPENSIVE = True to enable; no model calls are made.")

Pinned-annotation replay skipped. Set RUN_EXPENSIVE = True to enable; no model calls are made.


## 6. Interpretation

Choose a method for a workload, not from an isolated overall score. The full
results favor IRCoT on MuSiQue multi-hop synthesis but favor the simpler dense
arm on NQ. A higher retrieval metric need not imply better answers.
Keep conclusions tied to measured cells and their selected question IDs.

Write a short experiment conclusion with:

1. Corpus, annotated complexity, exact matched denominator, and missing cells.
2. Answer quality and paired interval, plus evidence/citation/failure diagnostics.
3. Recorded usage, cache warmth, and any unknown construction/execution cost.
4. One limitation and one controlled follow-up that could change your conclusion.

[The starter notebook](99_starter_agent.ipynb) turns this evaluation discipline
into an editable agent experiment.

Keep gold answers and hop graphs with the scorer. Local pinned embeddings and
the Vector proxy are used for live RI #2 experiments. Source IDs establish
provenance; they do not by themselves establish factual entailment.

See the [participant guide](GUIDE.md), [corpus/provenance record](CORPUS.md),
[experiment contracts](docs/EXPERIMENTS.md), and [full matched comparison](docs/COMPARISONS.md).